# Sample poses from a trained model

Pick any trained model of the model zoo (RSGM runs, flows converted from `SO3PriorTrainer`, ...), draw poses from it and look at them, next to real poses of the test split.

**Rendering.** Meshes are drawn with PyVista (`hp.visualization.pv_render`, offscreen; the pyrender backend of body_visualizer does not run on macOS). That needs the SMPL-X body model `smplx/SMPLX_NEUTRAL.npz` in the `body_models` folder of `config/paths.yaml` (download it from https://smpl-x.is.tue.mpg.de/downloads; it is licensed, so it is not in the repository). Without it the poses are drawn as skeletons (forward kinematics of SMPL, which only needs `smpl_skeleton.npz`), from the front and from the side.

The models speak 19 joint rotations (`AmassJoints4Learning`); for the body model the two toe joints, which are not in the data (always the rest pose in our AMASS), are set to the rest pose.

Sections: 1. the models there are · 2. one model: sample, log-likelihood, pictures · 3. real poses for comparison · 4. an interactive panel.

In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import roma
import torch
from omegaconf import OmegaConf

from hp.data.amass_cfg import AmassJoints4Learning
from hp.evaluation.data import load_reference
from hp.models.io import MANIFEST, load_model
from hp.utils.paths import get_path
from hp.utils.skeleton_forward_kinematics import SkelForwardKinematics, perform_forward_kinematics

# ---- parameters ----------------------------------------------------------------------
EXPERIMENTS = "~/data/experiments"      # searched for trained models (folders with model.json)
INCLUDE_TRIALS = False                  # also list the trials of the hyperparameter search and the checkpoints
MODEL = "rsgm_tuned_t23/run_0001"       # a folder below EXPERIMENTS (see the table of section 1), or an absolute path
WEIGHTS = "weights_ema"                 # weights_ema (if the run has them) or weights
N_SAMPLES = 8
SEED = 0
SAMPLE_STEPS = None                     # steps of the sampler (RSGM); None: the model's own setting
LIKELIHOOD_STEPS = 200                  # ODE steps of RSGM's log-likelihood; 0: do not compute it
DEVICES = {"rsgm": "mps", "so3_flow": "cpu"}   # by model type; others: cpu
VIEWS = (0, 90)                         # azimuths (degrees) of the mesh renders; skeletons: front and side
TEST_ROTATIONS = "~/data/AMASS_rotations"     # real poses for comparison (the official test split)
# ----------------------------------------------------------------------------------------

expand = lambda p: Path(p).expanduser()
EXPERIMENTS = expand(EXPERIMENTS)

## 1. The models there are

Every folder below `EXPERIMENTS` with a `model.json` (a checkpoint of the model zoo), with its type, size and the data it was trained on (from its `config.yaml`, if it is a run of `scripts/train_model.py`).

In [ ]:
def model_table(root, include_trials=False):
    rows = []
    for manifest in sorted(root.rglob(MANIFEST)):
        folder = manifest.parent
        rel = folder.relative_to(root)
        if not include_trials and ("checkpoint_" in str(rel) or str(rel).startswith("rsgm_hpo")):
            continue
        args = json.loads(manifest.read_text())["build_args"]
        config = folder / "config.yaml"
        data, steps = "-", "-"
        if config.is_file():
            c = OmegaConf.load(config)
            data = str(c.get("data", {}).get("rotations", "-"))
            steps = c.get("train", {}).get("max_iterations", "-")
        hidden = args.get("hidden")
        rows.append(dict(model=str(rel), type=args["name"], net=f"{hidden[0]} x {len(hidden)}" if hidden else "-",
                         trained_on=data.replace("~/data/", ""), steps=steps,
                         ema=(folder / "weights_ema.pt").is_file()))
    return pd.DataFrame(rows)


models = model_table(EXPERIMENTS, INCLUDE_TRIALS)
pd.set_option("display.width", 200)
print(models.to_string(index=False))

## 2. Sample from one model

Rendering helpers first: the 19 rotations of a pose become the 63 axis-angle numbers of the 21 body joints of SMPL-X (`pose_body`), which the body model turns into a mesh. Without the body model, the joints of SMPL's skeleton are drawn instead.

In [ ]:
PARENTS = SkelForwardKinematics().parents[:22]
BODY_MODEL = Path(get_path("body_models")) / "smplx" / "SMPLX_NEUTRAL.npz"


def to_pose_body(poses):
    # [N, 19, 3, 3] rotation matrices -> [N, 63]: axis-angle of the 21 body joints, the toes (10, 11) at rest
    n = len(poses)
    pose_body = torch.zeros(n, 21, 3)
    pose_body[:, [j.value - 1 for j in AmassJoints4Learning]] = roma.rotmat_to_rotvec(poses.float().cpu())
    return pose_body.reshape(n, 63)


renderer = None
if BODY_MODEL.is_file():
    from hp.visualization.render_tools import HumanRender
    renderer = HumanRender()
    print(f"meshes with PyVista, body model {BODY_MODEL}")
else:
    print(f"no body model at {BODY_MODEL}: skeletons instead of meshes")


def draw_skeleton(ax, joints, view, color="C0"):
    a, b = (0, 1) if view == "front" else (2, 1)
    for j in range(1, 22):
        p = PARENTS[j]
        ax.plot([joints[j, a], joints[p, a]], [joints[j, b], joints[p, b]], color=color, lw=2.2, solid_capstyle="round")
    ax.scatter(joints[:22, a], joints[:22, b], s=6, color=color)
    ax.set_aspect("equal")
    ax.set_xlim(-1.0, 1.0)
    ax.set_ylim(-1.25, 0.85)
    ax.axis("off")


def show_poses(poses, titles=None, color="C0", suptitle=None):
    # Draw poses [N, 19, 3, 3]: meshes (one row per view) if the body model is there, else skeletons (front, side).
    n = len(poses)
    titles = titles or [""] * n
    if renderer is not None:
        images = renderer.render_images(to_pose_body(poses), azimuths_deg=VIEWS)  # [N, views, h, w, 3]
        rows = len(VIEWS)
        fig, axes = plt.subplots(rows, n, figsize=(1.9 * n, 2.3 * rows), squeeze=False)
        for i in range(n):
            for r in range(rows):
                axes[r, i].imshow(images[i, r])
                axes[r, i].axis("off")
            axes[0, i].set_title(titles[i], fontsize=8)
    else:
        joints = perform_forward_kinematics(poses.float().cpu(), AmassJoints4Learning).numpy()
        fig, axes = plt.subplots(2, n, figsize=(1.6 * n, 4.6), squeeze=False)
        for i in range(n):
            draw_skeleton(axes[0, i], joints[i], "front", color)
            draw_skeleton(axes[1, i], joints[i], "side", color)
            axes[0, i].set_title(titles[i], fontsize=8)
        axes[0, 0].text(-1.0, 0.95, "front", fontsize=8, color="0.4")
        axes[1, 0].text(-1.0, 0.95, "side", fontsize=8, color="0.4")
    if suptitle:
        fig.suptitle(suptitle, fontsize=10)
    fig.tight_layout()
    plt.show()


def load(name, weights=WEIGHTS):
    folder = Path(name) if Path(name).is_absolute() else EXPERIMENTS / name
    kind = json.loads((folder / MANIFEST).read_text())["build_args"]["name"]
    if not (folder / f"{weights}.pt").is_file():
        weights = "weights"
    return load_model(folder, device=DEVICES.get(kind, "cpu"), weights=weights), weights


def sample(model, n, seed, sample_steps=None, likelihood_steps=LIKELIHOOD_STEPS):
    # n poses from the model, and their log-likelihoods (None if not computed). Returns poses on the cpu.
    if sample_steps is not None and hasattr(model, "sample_steps"):
        model.sample_steps = sample_steps
    torch.manual_seed(seed)
    start = time.perf_counter()
    with torch.no_grad():
        poses = model.sample(n).float().cpu()
    seconds = time.perf_counter() - start
    ll = None
    if likelihood_steps and model.has_log_prob:
        if hasattr(model, "likelihood_steps"):
            model.likelihood_steps = likelihood_steps
        with torch.no_grad():
            ll = model.log_prob(poses.to(model.device)).float().cpu().numpy()
    return poses, ll, seconds

In [ ]:
model, used_weights = load(MODEL)
poses, ll, seconds = sample(model, N_SAMPLES, SEED, SAMPLE_STEPS)
print(f"{MODEL} ({model.name}, {model.num_parameters():,} parameters, {used_weights}): {N_SAMPLES} poses in {seconds:.2f} s")
titles = [f"#{i}" + (f"\nlog p {ll[i]:.0f}" if ll is not None else "") for i in range(N_SAMPLES)]
show_poses(poses, titles, color="C1", suptitle=f"samples of {MODEL}, seed {SEED}")
if ll is not None:
    print("log-likelihood of the samples:", np.round(ll, 1), f"(mean {ll.mean():.1f})")

## 3. Real poses for comparison

Random poses of the official test split (Transitions + SSM, all frames), with their log-likelihood under the chosen model.

In [ ]:
test = load_reference("test", TEST_ROTATIONS, "original")
real = test[torch.randperm(len(test), generator=torch.Generator().manual_seed(SEED))[:N_SAMPLES]]
real_ll = None
if LIKELIHOOD_STEPS and model.has_log_prob:
    with torch.no_grad():
        real_ll = model.log_prob(real.to(model.device)).float().cpu().numpy()
titles = [f"test" + (f"\nlog p {real_ll[i]:.0f}" if real_ll is not None else "") for i in range(N_SAMPLES)]
show_poses(real, titles, color="C0", suptitle=f"real test poses, log p under {MODEL}")

## 4. Interactive panel

Choose a model, the number of poses, the seed and the sampler's steps, and press **Sample**. Models are loaded once and kept.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

cache = {}
model_box = widgets.Dropdown(options=list(models.model), value=MODEL if MODEL in list(models.model) else None,
                             description="model", layout=widgets.Layout(width="520px"))
n_box = widgets.IntSlider(value=N_SAMPLES, min=1, max=16, description="poses")
seed_box = widgets.IntText(value=SEED, description="seed", layout=widgets.Layout(width="160px"))
steps_box = widgets.IntText(value=100, description="steps", layout=widgets.Layout(width="160px"))
ll_box = widgets.Checkbox(value=True, description="log-likelihood")
button = widgets.Button(description="Sample", button_style="primary")
out = widgets.Output()


def on_sample(_):
    with out:
        out.clear_output(wait=True)
        if model_box.value is None:
            print("choose a model")
            return
        if model_box.value not in cache:
            print(f"loading {model_box.value} ...")
            cache[model_box.value] = load(model_box.value)[0]
        m = cache[model_box.value]
        p, l, sec = sample(m, n_box.value, seed_box.value, steps_box.value, LIKELIHOOD_STEPS if ll_box.value else 0)
        print(f"{model_box.value}: {n_box.value} poses in {sec:.2f} s" + (f", mean log p {l.mean():.1f}" if l is not None else ""))
        show_poses(p, [f"#{i}" + (f"\nlog p {l[i]:.0f}" if l is not None else "") for i in range(len(p))], color="C1")


button.on_click(on_sample)
display(widgets.VBox([model_box, widgets.HBox([n_box, seed_box, steps_box, ll_box, button]), out]))